# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [9]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

Africa

Africa had the fastest population growth between 1970 and 2022. Its population increased by around 290.4%. Oceania grew by 131.2%, while Europe had the slowest growth at 13.3%.

This shows that Africa's population has grown much faster than the other continents.


I used a line chart because it is easy to see how the population changed over the years. I compared each continent with its population in 1970 so that all continents could be compared fairly. I used different colours to make each line easy to identify.

In [10]:

# Q1 - Which continent is growing fastest?

continent_pop = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

baseline = (
    continent_pop[continent_pop['Year'] == 1970]
    [['Continent', 'Population']]
    .rename(columns={'Population': 'Population_1970'})
)

continent_pop = continent_pop.merge(
    baseline, on='Continent'
)

continent_pop['Growth (%)'] = (
    (continent_pop['Population'] /
     continent_pop['Population_1970']) - 1
) * 100

colours = {
    'Africa': '#E69F00',
    'Asia': '#0072B2',
    'Europe': '#6A3D9A',
    'North America': '#56B4E9',
    'Oceania': '#666666',
    'South America': '#CC79A7'
}

fig = px.line(
    continent_pop,
    x='Year',
    y='Growth (%)',
    color='Continent',
    color_discrete_map=colours,
    markers=True,
    title='Africa Has Experienced the Fastest Population Growth Since 1970',
    template='plotly_white'
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Population growth since 1970 (%)',
    legend_title='Continent'
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

print(
    continent_pop[continent_pop['Year'] == 2022]
    [['Continent', 'Growth (%)']]
    .sort_values('Growth (%)', ascending=False)
)


        Continent  Growth (%)
7          Africa  290.409905
39        Oceania  131.200871
47  South America  126.391835
15           Asia  120.120725
31  North America   90.307634
23         Europe   13.297813


---
## Q2 — Which countries added the most people?
The 15 countries that added the most people between 1970 and 2022 are:

India

China

Pakistan

Nigeria

Indonesia

United States

Brazil

Bangladesh

Ethiopia

DR Congo

Philippines

Mexico

Egypt

Iran

Vietnam

Out of these 15 countries, 4 are in Africa: Nigeria, Ethiopia, DR Congo and Egypt.

India added the most people, followed by China.

I used a horizontal bar chart because it makes it easy to compare all 15 countries. I arranged the bars by population increase so the countries with the biggest increases are easy to spot. I used a separate colour for African countries to make them stand out.



In [11]:

# Q2 - Which countries added the most people?

df['Population Change'] = (
    df['2022 Population'] - df['1970 Population']
)

top15 = (
    df.nlargest(15, 'Population Change')
    .sort_values('Population Change')
    .copy()
)

top15['Group'] = np.where(
    top15['Continent'] == 'Africa',
    'Africa',
    'Other continents'
)

fig = px.bar(
    top15,
    x='Population Change',
    y='Country',
    orientation='h',
    color='Group',
    color_discrete_map={
        'Africa': '#E69F00',
        'Other continents': '#999999'
    },
    text='Population Change',
    title='India and China Added the Most People Between 1970 and 2022',
    template='plotly_white'
)

fig.update_traces(
    texttemplate='%{x:,.0f}',
    textposition='outside',
    cliponaxis=False
)

fig.update_layout(
    xaxis_title='People added (1970–2022)',
    yaxis_title='',
    legend_title=''
)

fig.update_xaxes(showgrid=False)

fig.show()

africa_count = (
    top15['Continent'] == 'Africa'
).sum()

print("African countries in the top 15:", africa_count)

print(
    top15[['Country', 'Continent', 'Population Change']]
    .sort_values('Population Change', ascending=False)
)


African countries in the top 15: 4
           Country      Continent  Population Change
92           India           Asia          859671872
41           China           Asia          603352887
156       Pakistan           Asia          176533990
149        Nigeria         Africa          162971948
93       Indonesia           Asia          160272945
221  United States  North America          137961517
27          Brazil  South America          118943623
16      Bangladesh           Asia          103644512
63        Ethiopia         Africa           95071678
55        DR Congo         Africa           78858479
163    Philippines           Asia           78123423
131         Mexico  North America           77214819
57           Egypt         Africa           76208117
94            Iran           Asia           60100865
228        Vietnam           Asia           56258007


---
## Q3 — Do bigger countries have more people?

In general, countries with larger land areas tend to have larger populations. But this is not the case for every country.

India and China have very large populations. Bangladesh and Singapore also have high populations compared with their small land areas. Greenland and Mongolia have large areas but much smaller populations.

So, having more land does not always mean having more people.

I used a scatter plot because it helps compare land area and population at the same time. I used different colours for each continent so I could see the differences between them. I also used logarithmic scales because the countries have very different land areas and population sizes.

In [12]:

# Q3 - Do bigger countries have more people?

fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    color_discrete_map=colours,
    hover_name='Country',
    log_x=True,
    log_y=True,
    title='Population Is Not Proportional to Land Area',
    template='plotly_white'
)

fig.update_traces(
    marker=dict(size=9, opacity=0.75)
)

# Highlight unusual observations
highlight_countries = [
    'Singapore',
    'Bangladesh',
    'India',
    'China',
    'Greenland',
    'Mongolia',
    'Australia'
]

for country in highlight_countries:
    r = df[df['Country'] == country].iloc[0]

    fig.add_annotation(
        x=r['Area'],
        y=r['2022 Population'],
        text=country,
        showarrow=True,
        arrowhead=2,
        font=dict(size=10)
    )

fig.update_layout(
    xaxis_title='Land area (km², logarithmic scale)',
    yaxis_title='2022 population (logarithmic scale)',
    legend_title='Continent'
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()


---
## Q4 — How crowded are the continents?
Population density is different across the continents. Based on the median values in the dataset:

North America: 221.4 people per km²

Asia: 123.0 people per km²

Europe: 106.9 people per km²

Oceania: 78.2 people per km²

Africa: 59.7 people per km²

South America: 17.8 people per km²

North America has the highest median density in this dataset, while South America has the lowest. There are also big differences between countries within the same continent.

I used a box plot because it helps me see how population density differs between countries in each continent. It shows the middle value and how much the density varies, instead of showing only the average. I used a logarithmic scale because some countries have much higher population density than others, making the differences easier to see.


In [13]:

# Q4 - How crowded are the continents?

fig = px.box(
    df,
    x='Continent',
    y='Density',
    color_discrete_sequence=['#0072B2'],
    points='outliers',
    log_y=True,
    title='Population Density Varies Widely Within Each Continent',
    template='plotly_white'
)

fig.update_layout(
    xaxis_title='Continent',
    yaxis_title='Population density (people per km², log scale)'
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

print(
    df.groupby('Continent')['Density']
    .median()
    .sort_values(ascending=False)
)


Continent
North America    221.42110
Asia             122.96515
Europe           106.86540
Oceania           78.24840
Africa            59.70940
South America     17.78870
Name: Density, dtype: float64


---
## Q5 — Where did populations grow — and where did they shrink?

Between 2000 and 2022, most countries in the dataset had population growth.

200 countries had population growth.

34 countries had population decline.

Qatar had the highest increase, around 317.2%.

Northern Mariana Islands had the biggest decrease, around 38.3%.

Overall, more countries experienced population growth than population decline.

I used a choropleth map because it shows population changes on the world map. This makes it easier to see which countries had growth and which had a decline. I used different colours for increases and decreases so the changes are easy to understand.

In [14]:

# Q5 - Where did populations grow and shrink?

df['Population Change (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population']
) * 100

fig = px.choropleth(
    df,
    locations='CCA3',
    locationmode='ISO-3',
    color='Population Change (%)',
    hover_name='Country',
    color_continuous_scale='PuOr',
    color_continuous_midpoint=0,
    title='Population Growth and Decline Across Countries, 2000–2022',
    template='plotly_white'
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True
    ),
    coloraxis_colorbar=dict(
        title='Population change (%)'
    )
)

fig.show()

print(
    "Countries with population growth:",
    (df['Population Change (%)'] > 0).sum()
)

print(
    "Countries with population decline:",
    (df['Population Change (%)'] < 0).sum()
)

print(
    df.nlargest(5, 'Population Change (%)')
    [['Country', 'Population Change (%)']]
)


Countries with population growth: 200
Countries with population decline: 34
                      Country  Population Change (%)
167                     Qatar             317.242239
219      United Arab Emirates             188.249439
59          Equatorial Guinea             144.520327
215  Turks and Caicos Islands             143.827358
148                     Niger             125.490255


In 2022, the world population in the dataset was around 7.97 billion. Asia had the largest share of the population, followed by Africa and Europe.

Asia: 59.2%

Africa: 17.9%

Europe: 9.3%

North America: 7.5%

South America: 5.5%

Oceania: 0.6%

China and India had the largest populations among individual countries. Asia alone had almost 60% of the world's population.

I used a treemap because it shows both continents and their countries in one chart. The bigger the box, the larger the population. This makes it easy to compare which continents and countries have more people.

In [15]:

# Q6 - Where do the world's 8 billion people live?

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_map=colours,
    title='Asia Contains the Largest Share of the World Population',
    template='plotly_white'
)

fig.update_traces(
    textinfo='label+percent parent',
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Population: %{value:,}<br>'
        '<extra></extra>'
    )
)

fig.update_layout(
    margin=dict(t=60, l=10, r=10, b=10)
)

fig.show()

total_population = df['2022 Population'].sum()

asia_population = df.loc[
    df['Continent'] == 'Asia',
    '2022 Population'
].sum()

print("Total world population:", total_population)

print(
    "Asia's share of world population:",
    round(asia_population / total_population * 100, 2),
    "%"
)


Total world population: 7973413042
Asia's share of world population: 59.21 %


The population shares changed between 1970 and 2022 as follows:

Region

1970

2022

Africa

9.89%

17.89%

Asia

58.06%

59.21%

Europe

17.76%

9.32%

Rest of the world

14.29%

13.57%

Africa's share increased the most, from 9.89% to 17.89%. Asia's share stayed almost the same, while Europe's share decreased significantly. The rest of the world's share also decreased slightly.



I used a 100% stacked bar chart because it makes it easy to compare the population shares in 1970 and 2022. Each bar represents the total world population, so I can clearly see how each region's share changed. I used the same colours for each region in both years to make the comparison easier.

In [16]:

# Q7 - Whose share of the world is rising?

shares = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

shares['Group'] = np.where(
    shares['Continent'].isin(
        ['Africa', 'Asia', 'Europe']
    ),
    shares['Continent'],
    'Rest of the world'
)

shares = (
    shares.groupby(['Group', 'Year'])['Population']
    .sum()
    .reset_index()
)

shares['Share (%)'] = (
    shares['Population'] /
    shares.groupby('Year')['Population'].transform('sum')
) * 100

share_colours = {
    'Africa': '#E69F00',
    'Asia': '#0072B2',
    'Europe': '#6A3D9A',
    'Rest of the world': '#999999'
}

fig = px.bar(
    shares,
    x='Year',
    y='Share (%)',
    color='Group',
    barmode='stack',
    color_discrete_map=share_colours,
    category_orders={
        'Group': [
            'Africa',
            'Asia',
            'Europe',
            'Rest of the world'
        ]
    },
    title='Africa’s World Population Share Is Rising While Europe’s Is Declining',
    template='plotly_white'
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Share of world population (%)',
    legend_title='Region',
    yaxis=dict(range=[0, 100])
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

print(
    shares[shares['Year'].isin([1970, 2022])]
    .pivot(
        index='Group',
        columns='Year',
        values='Share (%)'
    )
    .round(2)
)


Year                1970   2022
Group                          
Africa              9.89  17.89
Asia               58.06  59.21
Europe             17.76   9.32
Rest of the world  14.29  13.57
